# Generalized quantum signal processing

Adapted from Qualtran v0.7.0. Copyright 2023 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from typing import cast

import numpy as np
import torch
from IPython import display as notebook_display

from flagquantum import Circuit
from flagquantum.algorithms.primitives.generalized_qsp import (
    GeneralizedQSP,
    source_gqsp_signature,
)
from flagquantum.algorithms.primitives.generalized_qsp_model import (
    source_generalized_qsp_cost,
    source_generalized_qsp_graph,
)
from flagquantum.compiler._component_trace.resource_model import (
    ComponentCost,
    GateCost,
    estimate_component_resources,
)
from flagquantum.compiler.registers.primitives.pauli_rotation import pauli_power_ir
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def unitary_x(
    signal: int,
    ports: Mapping[str, tuple[int, ...]],
    workspace: tuple[int, ...],
    value: int | None,
    adjoint: bool,
) -> tuple[Instruction, ...]:
    flip = (Instruction("x", (signal,)),) if value == 0 else ()
    return (
        *flip,
        *pauli_power_ir(
            "x",
            ports["q"][0],
            -1 if adjoint else 1,
            control=None if value is None else signal,
        ),
        *flip,
    )


def show_signatures(items: Sequence[str]) -> None:
    signature = source_gqsp_signature(Signature((Register("q", quantum_bit_type()),)))
    for name in items:
        display(SVG(render_call_graph_svg(export_signature_graph_dot(signature, name))))


## `GeneralizedQSP`
Applies a QSP polynomial $P$ to a unitary $U$ to obtain a block-encoding of $P(U)$.

Given a unitary $U$ and a QSP polynomial $P$ (and its complementary polynomial $Q$),
this gate implements the following unitary:

$$
    \begin{bmatrix} P(U) & \cdot \\ Q(U) & \cdot \end{bmatrix}
$$

The polynomials $P$ and $Q$ should satisfy:

$$
    \left|P(e^{i \theta})\right|^2 + \left|Q(e^{i \theta})\right|^2 = 1 ~~\text{for every}~ \theta \in \mathbb{R}
$$

The polynomial $P$ is said to be a QSP Polynomial if it satisfies:

$$
    \left|P(e^{i \theta})\right|^2 \le 1 ~~\text{for every}~ \theta \in \mathbb{R}
$$

If only the QSP polynomial $P$ is known, one can simply call
`GeneralizedQSP.from_qsp_polynomial(unitary_layout, unitary_emitter, P)` which automatically computes $Q$.

### Using Laurent Polynomials
To apply GQSP with the transformation given by $P'$

$$
P(z) = \sum_{n = -a}^b p_n z^n
$$

where $a, b \ge 0$, we can simply invoke GQSP with the standard polynomial $P'(z) = z^a P(z)$
which has degree $a + b$, and pass `negative_power=a`.

Given complementary QSP polynomials $P', Q'$ and `negative_power=a`,
this gate implements the unitary transform:

$$
    \begin{bmatrix} U^{-a} P'(U) & \cdot \\ U^{-a} Q'(U) & \cdot \end{bmatrix}
$$


The exact circuit implemented by this gate is described in Figure 2.

#### Parameters
 - `unitary_layout`, `unitary_emitter`: Ordered native register ports and a caller-owned coherent unitary implementation.
 - `P`: Co-efficients of a complex QSP polynomial.
 - `Q`: Co-efficients of a complex QSP polynomial.
 - `negative_power`: value of $k$, which effectively applies $z^{-k} P(z)$. defaults to 0.
 - `precision`: The error in the synthesized unitary. This is used to compute the required precision for each single qubit SU2 rotation. 

#### References
 - [Generalized Quantum Signal Processing](https://arxiv.org/abs/2308.01501). Motlagh and Wiebe. (2023). Theorem 3; Figure 2; Theorem 6.


In [ ]:
# GeneralizedQSP composes the accepted numerical and Compiler owners.


### Example Instances

In [ ]:
gqsp = GeneralizedQSP.from_qsp_polynomial((("q", 1),), unitary_x, (0.5, 0.5))


In [ ]:
gqsp_with_negative_power = GeneralizedQSP.from_qsp_polynomial(
    (("q", 1),), unitary_x, (0.5, 0, 0.5), negative_power=1
)


In [ ]:
gqsp_with_large_negative_power = GeneralizedQSP.from_qsp_polynomial(
    (("q", 1),), unitary_x, (0.5, 0, 0.5), negative_power=5
)


#### Graphical Signature

In [ ]:
show_signatures(("gqsp", "gqsp_with_negative_power", "gqsp_with_large_negative_power"))


### Call Graph

In [ ]:
gqsp_graph = source_generalized_qsp_graph(
    2,
    root="GeneralizedQSP",
    signal_rotation="SU2Rotation",
    zero_control_unitary="OpenControlledX",
    unitary_adjoint="XAdjoint",
    controlled_adjoint="ControlledXAdjoint",
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(gqsp_graph.root, gqsp_graph.direct_calls)
        )
    )
)
display(gqsp_graph.leaf_counts)


## Append and independent resource accounts

The examples retain the original X unitary, coefficient pairs and Laurent offsets. The caller owns physical wire binding. Continuous signal rotations retain absolute phase. The numerical complement and declared model are distinct from emitted primitives. This example borrows zero scratch wires, has one unitary branch and retains one zero idle qubit.


In [ ]:
initial = np.array([1, 2j, -1j, 2], dtype=complex) / np.sqrt(10)
circuit = Circuit(
    3, inputs=torch.tensor(np.kron(initial, [1, 0])), dtype=torch.complex128
)
circuit.append(gqsp.make_on(signal=0, target=(1,)))
complete_state = circuit.state().detach().numpy().reshape(4, 2)
np.testing.assert_array_equal(complete_state[:, 1], 0)
primitive_counts = dict(Counter(g.name for g in circuit.to_ir().instructions))
model = source_generalized_qsp_cost(
    2,
    1,
    signal_rotation=ComponentCost("rotation", 1, GateCost(rotation=3)),
    zero_control_unitary=ComponentCost("open-X", 2, GateCost(clifford=3)),
    unitary_adjoint=ComponentCost("X-adjoint", 1, GateCost(clifford=1)),
    controlled_adjoint=ComponentCost("controlled-X-adjoint", 2, GateCost(clifford=1)),
)
modeled_resources = estimate_component_resources(model)
display(primitive_counts)
display(modeled_resources)
